# Homework 2: End-to-End Reconstruction Before Generative Models

This notebook implements the complete end-to-end reconstruction pipeline requested in the homework. Starting from clean Mayo images, it generates synthetic degraded measurements

$$
    y^\delta = Kx + e,
$$

where $K$ is a known motion-blur operator and $e$ is additive Gaussian noise. The trained models reconstruct the clean image from the corrupted measurement and are compared visually and quantitatively.

The notebook is designed to run on Google Colab from the Drive directory:

```text
/content/drive/MyDrive/LM_INFORMATICA/COMPUTATIONAL_IMAGING/notebooks
```

All heavy artifacts are saved outside the notebook, under the project Drive directory.

## Homework Goals and Deliverables

The completed notebook includes:

1. dataset preparation and synthetic corruption;
2. implementation of `SimpleCNN`, `ResCNN`, and the optional `UNet` extension;
3. supervised end-to-end training with online corruption generation;
4. saved model weights in `../weights/`;
5. training curves;
6. visual and quantitative reconstruction comparison;
7. final written discussion.

The implementation follows the lecture material on PyTorch training mechanics, CNNs, residual learning, UNet-style encoder-decoder models, and evaluation of output images.

## Environment Setup

Run this cell first. It mounts Google Drive, sets the project paths, imports the required libraries, and loads IPPy from the course material folder.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import csv
import glob
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import torch
from IPython.display import Image as DisplayImage
from IPython.display import display
from PIL import Image
from skimage.metrics import (
    mean_squared_error,
    peak_signal_noise_ratio,
    structural_similarity,
)
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from tqdm.auto import tqdm

PROJECT_ROOT = Path("/content/drive/MyDrive/LM_INFORMATICA/COMPUTATIONAL_IMAGING")
MAYO_DIR = PROJECT_ROOT / "Mayo"
TRAIN_DIR = MAYO_DIR / "train"
TEST_DIR = MAYO_DIR / "test"
IPPY_DIR = PROJECT_ROOT / "IPPy"
WEIGHTS_DIR = PROJECT_ROOT / "weights" / "hw2"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "hw2"

for directory in [WEIGHTS_DIR, OUTPUT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

sys.path.append(str(PROJECT_ROOT))
sys.path.append(str(IPPY_DIR))

from IPPy import operators, utilities

device = utilities.get_device()
torch.manual_seed(0)

print("Working device:", device)
print("Project root:", PROJECT_ROOT)
print("Weights directory:", WEIGHTS_DIR)
print("IPPy directory:", IPPY_DIR)

In [ ]:
DATA_SHAPE = 256
BATCH_SIZE = 8
NUM_WORKERS = 0
KERNEL_SIZE = 15
MOTION_ANGLE = 30.0
NOISE_LEVEL = 0.01
NUM_EPOCHS = 20
LEARNING_RATE = 1e-3

print("Training directory:", TRAIN_DIR)
print("Test directory:", TEST_DIR)
print("Noise level:", NOISE_LEVEL)
print("Epochs:", NUM_EPOCHS)

## Part 1: Data Pipeline and Synthetic Measurements

The dataset returns grayscale Mayo slices resized to $256 \times 256$ and represented as tensors in $[0,1]$. The corrupted measurements are generated through the known operator $K$ and additive Gaussian noise.

In [ ]:
class MayoDataset(Dataset):
    def __init__(self, data_path, data_shape=256):
        super().__init__()
        self.fname_list = sorted(glob.glob(f"{data_path}/*/*.png"))
        self.transform = transforms.Compose(
            [
                transforms.ToTensor(),
                transforms.Resize((data_shape, data_shape), antialias=True),
            ]
        )

    def __len__(self):
        return len(self.fname_list)

    def __getitem__(self, idx):
        image_path = self.fname_list[idx]
        image = Image.open(image_path).convert("L")
        image = self.transform(image)
        return image


def add_gaussian_noise(image: torch.Tensor, noise_level: float) -> torch.Tensor:
    noisy = image + noise_level * torch.randn_like(image)
    return torch.clamp(noisy, 0.0, 1.0)


def save_clean_corrupted_pair(clean: torch.Tensor, corrupted: torch.Tensor, output_path: Path) -> None:
    output_path.parent.mkdir(parents=True, exist_ok=True)
    clean_image = clean[0, 0].detach().cpu()
    corrupted_image = corrupted[0, 0].detach().cpu()

    fig, axes = plt.subplots(1, 2, figsize=(8, 4), constrained_layout=True)
    axes[0].imshow(clean_image, cmap="gray", vmin=0.0, vmax=1.0)
    axes[0].set_title("Clean")
    axes[0].axis("off")
    axes[1].imshow(corrupted_image, cmap="gray", vmin=0.0, vmax=1.0)
    axes[1].set_title("Motion blur + noise")
    axes[1].axis("off")
    fig.savefig(output_path, dpi=150)
    plt.show()
    plt.close(fig)

In [ ]:
train_dataset = MayoDataset(TRAIN_DIR, DATA_SHAPE)
test_dataset = MayoDataset(TEST_DIR, DATA_SHAPE)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
)

clean = next(iter(train_loader)).to(device)
K = operators.Blurring(
    img_shape=(DATA_SHAPE, DATA_SHAPE),
    kernel_type="motion",
    kernel_size=KERNEL_SIZE,
    motion_angle=MOTION_ANGLE,
)
corrupted = add_gaussian_noise(K(clean), NOISE_LEVEL)

clean_corrupted_path = OUTPUT_DIR / "hw2_clean_corrupted.png"
save_clean_corrupted_pair(clean, corrupted, clean_corrupted_path)

print("Train images:   ", len(train_dataset))
print("Test images:    ", len(test_dataset))
print("Batch shape:    ", tuple(clean.shape))
print(f"Clean range:     [{clean.min().item():.4f}, {clean.max().item():.4f}]")
print(f"Corrupted range: [{corrupted.min().item():.4f}, {corrupted.max().item():.4f}]")
print("Saved figure:   ", clean_corrupted_path)

## Part 2: Reconstruction Networks

The mandatory architectures are `SimpleCNN` and `ResCNN`. The residual model predicts a correction and adds it back to the corrupted input. The optional `UNet` extension adds a multiscale encoder-decoder architecture with skip connections.

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, in_ch=1, out_ch=1, n_filters=32, kernel_size=3):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, n_filters, kernel_size, padding="same")
        self.conv2 = nn.Conv2d(n_filters, n_filters, kernel_size, padding="same")
        self.conv3 = nn.Conv2d(n_filters, out_ch, kernel_size, padding="same")
        self.relu = nn.ReLU()

    def forward(self, x):
        h = self.relu(self.conv1(x))
        h = self.relu(self.conv2(h))
        out = self.conv3(h)
        return out


class ResCNN(nn.Module):
    def __init__(self, in_ch=1, out_ch=1, n_filters=32, kernel_size=3):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, n_filters, kernel_size, padding="same")
        self.conv2 = nn.Conv2d(n_filters, n_filters, kernel_size, padding="same")
        self.conv3 = nn.Conv2d(n_filters, out_ch, kernel_size, padding="same")
        self.relu = nn.ReLU()
        self.tanh = nn.Tanh()

    def forward(self, x):
        h = self.relu(self.conv1(x))
        h = self.relu(self.conv2(h))
        out = self.tanh(self.conv3(h))
        return out + x

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1),
            nn.ReLU(),
        )

    def forward(self, x):
        return self.block(x)


class DownBlock(nn.Module):
    def __init__(self, in_ch, out_ch, block_cls=DoubleConv):
        super().__init__()
        self.pool = nn.MaxPool2d(2)
        self.block = block_cls(in_ch, out_ch)

    def forward(self, x):
        return self.block(self.pool(x))


class UpBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch, block_cls=DoubleConv):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_ch, out_ch, kernel_size=2, stride=2)
        self.block = block_cls(out_ch + skip_ch, out_ch)

    def forward(self, x, skip):
        x = self.up(x)
        if x.shape[-2:] != skip.shape[-2:]:
            x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
        x = torch.cat([skip, x], dim=1)
        return self.block(x)


class UNet(nn.Module):
    def __init__(self, in_ch=1, out_ch=1, base_ch=32):
        super().__init__()
        self.enc1 = DoubleConv(in_ch, base_ch)
        self.enc2 = DownBlock(base_ch, 2 * base_ch)
        self.enc3 = DownBlock(2 * base_ch, 4 * base_ch)
        self.bottleneck = DownBlock(4 * base_ch, 8 * base_ch)
        self.dec3 = UpBlock(8 * base_ch, 4 * base_ch, 4 * base_ch)
        self.dec2 = UpBlock(4 * base_ch, 2 * base_ch, 2 * base_ch)
        self.dec1 = UpBlock(2 * base_ch, base_ch, base_ch)
        self.out_conv = nn.Conv2d(base_ch, out_ch, kernel_size=1)

    def forward(self, x):
        s1 = self.enc1(x)
        s2 = self.enc2(s1)
        s3 = self.enc3(s2)
        h = self.bottleneck(s3)
        h = self.dec3(h, s3)
        h = self.dec2(h, s2)
        h = self.dec1(h, s1)
        return self.out_conv(h)

In [ ]:
def count_trainable_parameters(model: nn.Module) -> int:
    return sum(param.numel() for param in model.parameters() if param.requires_grad)


def build_reconstruction_models() -> dict[str, nn.Module]:
    return {
        "simplecnn": SimpleCNN(in_ch=1, out_ch=1, n_filters=32),
        "rescnn": ResCNN(in_ch=1, out_ch=1, n_filters=32),
        "unet": UNet(in_ch=1, out_ch=1, base_ch=32),
    }


models = build_reconstruction_models()
for name, model in models.items():
    print(f"{name}: {count_trainable_parameters(model)} trainable parameters")

## Part 3: Training, Saving, and Reloading the Models

The corrupted measurements are generated online during training. All models are trained under the same corruption setup, using MSE loss and Adam, then their final weights are saved in `../weights/hw2/` and reloaded before evaluation.

In [ ]:
def train_model(model, train_loader, K, weights_path, device, num_epochs=20, noise_level=0.01, lr=1e-3):
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()
    history = []

    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        num_samples = 0

        progress_bar = tqdm(train_loader, desc=f"Epoch {epoch + 1}/{num_epochs}")
        for clean in progress_bar:
            clean = clean.to(device)
            corrupted = add_gaussian_noise(K(clean), noise_level)

            optimizer.zero_grad()
            reconstruction = model(corrupted)
            loss = loss_fn(reconstruction, clean)
            loss.backward()
            optimizer.step()

            batch_size = clean.shape[0]
            running_loss += loss.item() * batch_size
            num_samples += batch_size
            progress_bar.set_postfix(loss=running_loss / num_samples)

        epoch_loss = running_loss / num_samples
        history.append(epoch_loss)

    weights_path = Path(weights_path)
    weights_path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(model.state_dict(), weights_path)
    return history


def save_training_curves(histories: dict[str, list[float]], output_path: Path) -> None:
    output_path.parent.mkdir(parents=True, exist_ok=True)

    fig, ax = plt.subplots(figsize=(6, 4), constrained_layout=True)
    for model_name, history in histories.items():
        epochs = range(1, len(history) + 1)
        ax.plot(epochs, history, marker="o", label=model_name)

    ax.set_xlabel("Epoch")
    ax.set_ylabel("Training MSE")
    ax.set_title("Training curves")
    ax.grid(True, alpha=0.3)
    ax.legend()
    fig.savefig(output_path, dpi=150)
    plt.show()
    plt.close(fig)

In [ ]:
histories = {}

for model_name, model in models.items():
    weights_path = WEIGHTS_DIR / f"hw2_{model_name}.pt"
    print(f"Training {model_name}")
    history = train_model(
        model=model,
        train_loader=train_loader,
        K=K,
        weights_path=weights_path,
        device=device,
        num_epochs=NUM_EPOCHS,
        noise_level=NOISE_LEVEL,
        lr=LEARNING_RATE,
    )
    model.load_state_dict(torch.load(weights_path, map_location=device))
    model.eval()
    histories[model_name] = history
    print("Saved and reloaded weights:", weights_path)

training_curves_path = OUTPUT_DIR / "hw2_training_curves.png"
save_training_curves(histories, training_curves_path)
print("Saved training curves:", training_curves_path)

## Part 4: Visual and Quantitative Comparison

The same corrupted test image is used as input for all trained models. This makes the comparison fair because all models receive exactly the same degraded measurement.

MSE, PSNR, and SSIM are computed using predefined functions from `skimage.metrics`, consistently with the evaluation section of the course.

In [ ]:
def compute_image_metrics(reconstruction: torch.Tensor, clean: torch.Tensor) -> dict[str, float]:
    reconstruction = torch.clamp(reconstruction.detach().cpu(), 0.0, 1.0)
    clean = torch.clamp(clean.detach().cpu(), 0.0, 1.0)

    clean_image = clean.squeeze().numpy()
    reconstruction_image = reconstruction.squeeze().numpy()

    mse = mean_squared_error(clean_image, reconstruction_image)
    psnr = peak_signal_noise_ratio(clean_image, reconstruction_image, data_range=1.0)
    ssim = structural_similarity(clean_image, reconstruction_image, data_range=1.0)
    return {"mse": mse, "psnr": psnr, "ssim": float(ssim)}


def save_metrics_table(metrics: dict[str, dict[str, float]], output_path: Path) -> None:
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with output_path.open("w", newline="") as file:
        writer = csv.writer(file)
        writer.writerow(["model", "mse", "psnr", "ssim"])
        for model_name, values in metrics.items():
            writer.writerow([model_name, values["mse"], values["psnr"], values["ssim"]])


def save_reconstruction_comparison(clean, corrupted, reconstructions, metrics, output_path: Path) -> None:
    output_path.parent.mkdir(parents=True, exist_ok=True)

    images = {"clean": clean, "corrupted": corrupted, **reconstructions}
    fig, axes = plt.subplots(1, len(images), figsize=(4 * len(images), 4), constrained_layout=True)

    for ax, (name, image) in zip(axes, images.items()):
        ax.imshow(torch.clamp(image[0, 0].detach().cpu(), 0.0, 1.0), cmap="gray", vmin=0.0, vmax=1.0)
        if name in metrics:
            ax.set_title(
                f"{name}\nMSE={metrics[name]['mse']:.4f}\n"
                f"PSNR={metrics[name]['psnr']:.2f}, SSIM={metrics[name]['ssim']:.3f}"
            )
        else:
            ax.set_title(name)
        ax.axis("off")

    fig.savefig(output_path, dpi=150)
    plt.show()
    plt.close(fig)

In [ ]:
@torch.no_grad()
def evaluate_reconstruction_models(models, test_loader, K, device, weights_dir, output_dir, noise_level):
    clean = next(iter(test_loader))[:1].to(device)
    corrupted = add_gaussian_noise(K(clean), noise_level)
    reconstructions = {}

    metrics = {
        "corrupted": compute_image_metrics(corrupted, clean),
    }

    for model_name, model in models.items():
        weights_path = weights_dir / f"hw2_{model_name}.pt"
        model.load_state_dict(torch.load(weights_path, map_location=device))
        model.to(device)
        model.eval()

        reconstruction = model(corrupted)
        reconstructions[model_name] = reconstruction
        metrics[model_name] = compute_image_metrics(reconstruction, clean)

    figure_path = output_dir / "hw2_reconstruction_comparison.png"
    metrics_path = output_dir / "hw2_metrics.csv"
    save_reconstruction_comparison(clean, corrupted, reconstructions, metrics, figure_path)
    save_metrics_table(metrics, metrics_path)

    for model_name, values in metrics.items():
        print(
            f"{model_name}: "
            f"MSE={values['mse']:.6f}, PSNR={values['psnr']:.2f}, SSIM={values['ssim']:.4f}"
        )
    print("Saved reconstruction comparison:", figure_path)
    print("Saved metrics table:", metrics_path)
    return metrics


metrics = evaluate_reconstruction_models(
    models=models,
    test_loader=test_loader,
    K=K,
    device=device,
    weights_dir=WEIGHTS_DIR,
    output_dir=OUTPUT_DIR,
    noise_level=NOISE_LEVEL,
)

In [ ]:
metrics_path = OUTPUT_DIR / "hw2_metrics.csv"
with metrics_path.open() as file:
    metrics_rows = list(csv.DictReader(file))
metrics_rows

## Saved Artifacts

These cells display the saved figures produced by the notebook. The weights are saved in `../weights/hw2/` with meaningful names.

In [ ]:
print("Weights:")
for path in sorted(WEIGHTS_DIR.glob("hw2_*.pt")):
    print(path)

print("\nFigures:")
print(training_curves_path)
print(OUTPUT_DIR / "hw2_reconstruction_comparison.png")

In [ ]:
display(DisplayImage(str(training_curves_path)))
display(DisplayImage(str(OUTPUT_DIR / "hw2_reconstruction_comparison.png")))

## Deliverables and Discussion

**Which model performed better in your experiments, and why do you think that happened?**

Complete this answer after inspecting the final metric table and the reconstruction comparison. In general, the best model should be selected by considering both numerical metrics and visual fidelity.

**Did the residual architecture help? If yes, in what sense?**

Compare `SimpleCNN` and `ResCNN`. The residual architecture is expected to help when the corrupted image already contains useful low-frequency structure, because the network can learn a correction rather than reconstructing the whole image from scratch.

**How did the noise level affect training and reconstruction quality?**

Higher noise levels make the inverse problem harder because the network must compensate for both blur and stochastic perturbations. This can increase the training loss and reduce reconstruction sharpness or metric values.

**Why is it important to generate the corruption through a known operator $K$ instead of treating the problem as a generic image-to-image task?**

Using a known operator keeps the reconstruction task connected to the inverse problem formulation. The corrupted image follows the model $y = Kx + e$, so training and evaluation remain consistent with the forward process that generated the data.

**Why should one be cautious when evaluating pure end-to-end methods only through visual quality?**

A visually pleasing reconstruction is not automatically reliable. In inverse problems, visual quality can hide artifacts, hallucinated structures, or failures caused by distribution shift. Quantitative metrics and knowledge of the forward operator are therefore necessary.

**If you implemented the optional UNet extension, how did it compare with the simpler CNN-based models?**

Complete this answer after comparing `UNet` against `SimpleCNN` and `ResCNN` in the metric table and visual figure. UNet has many more parameters and a multiscale structure, so it may improve detail recovery, but it can also require more training time and may overfit if the setup is not controlled.